# Milestone 3 - Retrieval-Augmented Generation (RAG)

## Setup: Knowledge Base, FAISS Index, and Zero-Shot Classifier

In [1]:
import subprocess, sys
try:
    import faiss
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
    import faiss

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Kaggle path with local fallback
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'train.csv'

train = pd.read_csv(DATA_PATH)
print(f'Loaded train.csv: {len(train)} rows')

# Build the knowledge base from the correct option of each row
print('Creating knowledge base')
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

# Embed the KB with all-MiniLM-L6-v2 and add to a FAISS L2 index
print('Loading embedding model and creating index')
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False).astype('float32')
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print('Knowledge base successfully created')
print(f'KB size: {len(kb)}, embedding dim: {kb_embeddings.shape[1]}')

# Zero-shot classifier shared by Q1, Q2, Q5, Q6, Q8
print('Loading bart-large-mnli zero-shot classifier (this downloads ~1.6 GB)...')
zs = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# Pre-build the row 150 inputs that several questions reuse
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']),
              str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])
print()
print(f'Row 150 prompt (first 80): {prompt_150[:80]}...')
print(f'Row 150 answer letter    : {row_150["answer"]}')
print(f'Row 150 correct option   : {ans_150[:80]}...')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 87.2 MB/s eta 0:00:00
Loaded train.csv: 2000 rows
Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Row 150 prompt (first 80): Select the most accurate option: What is the butterfly effect, as defined by Lor...
Row 150 answer letter    : C
Row 150 correct option   : The butterfly effect is the phenomenon that a small change in the initial condit...


### Helper: MAP@3

Same definition as Milestone 1: rank-1 hit = 1.0, rank-2 = 0.5, rank-3 = 0.333,
otherwise 0.0.

In [2]:
def map_at_3(truth, prediction):
    """MAP@3 for one question. prediction is a list of up to 3 letters."""
    if truth in prediction:
        return 1.0 / (prediction.index(truth) + 1)
    return 0.0

assert map_at_3('C', ['C', 'A', 'B']) == 1.0
assert map_at_3('B', ['D', 'B', 'E']) == 0.5
print('MAP@3 helper ready.')

MAP@3 helper ready.


## Q1. Zero-Shot Probability of the Ground-Truth Option (Row 150)

In [3]:
result_q1 = zs(prompt_150, candidate_labels=labels_150)

# The ground-truth label is the text of the correct option for row 150
gt_letter_q1 = row_150['answer']
gt_label_text = labels_150['ABCDE'.index(gt_letter_q1)]

# Locate that label in the model's output and pull its score
gt_position = result_q1['labels'].index(gt_label_text)
gt_score = result_q1['scores'][gt_position]

print(f'Top label (first 60): {str(result_q1["labels"][0])[:60]}...')
print(f'Top score            : {result_q1["scores"][0]}')
print(f'Ground-truth letter  : {gt_letter_q1}')
print(f'Ground-truth score   : {gt_score}')

q1_answer = round(gt_score, 3)
print()
print(f'ANSWER Q1: {q1_answer}')

Top label (first 60): The butterfly effect is the phenomenon that a small change i...
Top score            : 0.38441911339759827
Ground-truth letter  : C
Ground-truth score   : 0.38441911339759827

ANSWER Q1: 0.384


## Q2. FAISS Retrieval Rank of the True Document (Row 150, k=10)

In [4]:
# Embed the prompt and search the FAISS index
prompt_150_emb = model.encode([prompt_150], show_progress_bar=False).astype('float32')
K = 10
distances, retrieved_indices = index.search(prompt_150_emb, K)
retrieved_indices = retrieved_indices[0]  # flatten batch dim

print(f'Retrieved KB indices (top {K}): {retrieved_indices.tolist()}')
print(f'Distances             (top {K}): {distances[0].tolist()}')

true_kb_idx = 150  # KB index 150 == correct option of train row 150
if true_kb_idx in retrieved_indices:
    q2_answer = int(np.where(retrieved_indices == true_kb_idx)[0][0]) + 1
    print(f'True document (KB index {true_kb_idx}) found at rank: {q2_answer}')
else:
    q2_answer = None
    print(f'True document NOT in top {K}')

print()
print(f'ANSWER Q2: {q2_answer}')

Retrieved KB indices (top 10): [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
Distances             (top 10): [0.26394256949424744, 0.26394256949424744, 0.2664284110069275, 0.2664284110069275, 0.2686218023300171, 0.26993614435195923, 0.26993614435195923, 0.26993614435195923, 0.2831609845161438, 0.2871459722518921]
True document (KB index 150) found at rank: 10

ANSWER Q2: 10


## Q3. Cross-Encoder Reranking Rank of the True Document

In [5]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Pull the text of each retrieved document from the KB
docs_10 = [kb[i] for i in retrieved_indices]

# Build (prompt, document) pairs and score them
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)
print(f'Cross-encoder scores: {ce_scores}')

# Sort indices of docs_10 by CE score, descending
sorted_ce_order = np.argsort(ce_scores)[::-1]
reranked_kb_indices = retrieved_indices[sorted_ce_order]
print(f'Reranked KB indices: {reranked_kb_indices.tolist()}')

if true_kb_idx in reranked_kb_indices:
    q3_answer = int(np.where(reranked_kb_indices == true_kb_idx)[0][0]) + 1
    print(f'True document rank after reranking: {q3_answer}')
else:
    q3_answer = None

print()
print(f'ANSWER Q3: {q3_answer}')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder scores: [4.6602316 4.6602316 4.737523  4.737523  4.687043  4.752577  4.752577
 4.752577  4.7072477 4.758512 ]
Reranked KB indices: [150, 1906, 847, 1693, 1269, 1532, 168, 576, 1701, 663]
True document rank after reranking: 1

ANSWER Q3: 1


## Q4. Token Count of a 5-Chunk RAG String (Row 42)

In [6]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

# Retrieve top 5 docs
prompt_42_emb = model.encode([prompt_42], show_progress_bar=False).astype('float32')
_, idx5 = index.search(prompt_42_emb, 5)
idx5 = idx5[0]
print(f'Row 42 retrieved KB indices (top 5): {idx5.tolist()}')

# Concatenate the 5 docs with single spaces
docs_5 = [kb[i] for i in idx5]
concatenated = ' '.join(docs_5)

# Build the RAG string in the exact required format
rag_string = f'Context: {concatenated} Question: {prompt_42}'
print(f'RAG string length (chars): {len(rag_string)}')
print(f'RAG string (first 200): {rag_string[:200]}...')

# Tokenize without truncation
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
encoded = bert_tokenizer(rag_string, truncation=False)
q4_answer = len(encoded['input_ids'])
print(f'Total tokens (no truncation): {q4_answer}')
print()
print(f'ANSWER Q4: {q4_answer}')

Row 42 retrieved KB indices (top 5): [1581, 1760, 42, 241, 439]
RAG string length (chars): 1127
RAG string (first 200): Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combinati...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens (no truncation): 216

ANSWER Q4: 216


## Q5. Zero-Shot with the True Document as Context (Row 150)

In [7]:
true_doc_150 = kb[150]
rag_q5 = f'Context: {true_doc_150} Question: {prompt_150}'
print(f'RAG string (first 200): {rag_q5[:200]}...')

result_q5 = zs(rag_q5, candidate_labels=labels_150)

# Locate the ground-truth label and read its score
gt_pos_q5 = result_q5['labels'].index(gt_label_text)
gt_score_q5 = result_q5['scores'][gt_pos_q5]
print(f'Top label (first 60): {str(result_q5["labels"][0])[:60]}...')
print(f'Top score            : {result_q5["scores"][0]}')
print(f'Ground-truth score   : {gt_score_q5}')

q5_answer = round(gt_score_q5, 3)
print()
print(f'ANSWER Q5: {q5_answer}')

RAG string (first 200): Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in ...
Top label (first 60): The butterfly effect is the phenomenon that a small change i...
Top score            : 0.9894258379936218
Ground-truth score   : 0.9894258379936218

ANSWER Q5: 0.989


## Q6. Adversarial RAG - Unrelated Context (KB[999])

In [8]:
adversarial_doc = kb[999]
print(f'KB[999] (first 80): {adversarial_doc[:80]}...')

rag_q6 = f'Context: {adversarial_doc} Question: {prompt_150}'
result_q6 = zs(rag_q6, candidate_labels=labels_150)

gt_pos_q6 = result_q6['labels'].index(gt_label_text)
gt_score_q6 = result_q6['scores'][gt_pos_q6]
print(f'Top label (first 60): {str(result_q6["labels"][0])[:60]}...')
print(f'Top score            : {result_q6["scores"][0]}')
print(f'Ground-truth score   : {gt_score_q6}')

q6_answer = round(gt_score_q6, 3)
print()
print(f'ANSWER Q6: {q6_answer}')

KB[999] (first 80): A thought experiment in which a demon guards a microscopic trapdoor in a wall se...
Top label (first 60): The butterfly effect is the phenomenon that a small change i...
Top score            : 0.5289478302001953
Ground-truth score   : 0.5289478302001953

ANSWER Q6: 0.529


## Q7. Hit Rate over the First 100 Rows (Top 5 Retrieval)

In [9]:
N_Q7 = 100
hits = 0

for i in range(N_Q7):
    row = train.iloc[i]
    p = str(row['prompt'])
    correct_opt_text = str(row[row['answer']])

    p_emb = model.encode([p], show_progress_bar=False).astype('float32')
    _, retrieved = index.search(p_emb, 5)
    retrieved = retrieved[0]
    retrieved_docs = [kb[j] for j in retrieved]

    # Hit if the exact correct-option string appears as a substring of any retrieved doc
    if any(correct_opt_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = hits / N_Q7 * 100
q7_answer = round(hit_rate, 1)
print(f'Hits: {hits} / {N_Q7}')
print()
print(f'ANSWER Q7: {q7_answer}')

Hits: 73 / 100

ANSWER Q7: 73.0


## Q8. Full RAG Pipeline MAP@3 over the First 20 Rows

In [10]:
N_Q8 = 20
map3_scores = []

for i in range(N_Q8):
    row = train.iloc[i]
    p = str(row['prompt'])
    correct_letter = row['answer']
    options = [str(row[c]) for c in 'ABCDE']

    # Step 1: Retrieve top 5
    p_emb = model.encode([p], show_progress_bar=False).astype('float32')
    _, retrieved = index.search(p_emb, 5)
    retrieved = retrieved[0]
    docs = [kb[j] for j in retrieved]

    # Step 2: Rerank with cross-encoder, pick the best document
    pairs = [[p, d] for d in docs]
    ce_s = cross_encoder.predict(pairs)
    best_doc = docs[int(np.argmax(ce_s))]

    # Step 3: Build the RAG string
    rag_str = f'Context: {best_doc} Question: {p}'

    # Step 4: Zero-shot classify
    res = zs(rag_str, candidate_labels=options)

    # Step 5: Rank options by score, take top 3 letters, compute MAP@3
    ranked_indices = [options.index(lbl) for lbl in res['labels']]
    top3_letters = ['ABCDE'[idx] for idx in ranked_indices[:3]]
    score = map_at_3(correct_letter, top3_letters)
    map3_scores.append(score)
    print(f'  Row {i:>2}: truth={correct_letter}, pred={top3_letters}, score={score:.3f}')

avg_map3 = float(np.mean(map3_scores))
q8_answer = round(avg_map3, 3)
print()
print(f'Average MAP@3 over {N_Q8} rows: {avg_map3}')
print()
print(f'ANSWER Q8: {q8_answer}')


  Row  0: truth=B, pred=['B', 'D', 'A'], score=1.000
  Row  1: truth=A, pred=['A', 'E', 'C'], score=1.000
  Row  2: truth=C, pred=['C', 'D', 'B'], score=1.000
  Row  3: truth=B, pred=['B', 'D', 'A'], score=1.000
  Row  4: truth=A, pred=['A', 'B', 'C'], score=1.000


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Row  5: truth=C, pred=['B', 'C', 'A'], score=0.500
  Row  6: truth=E, pred=['E', 'B', 'D'], score=1.000
  Row  7: truth=A, pred=['A', 'B', 'C'], score=1.000
  Row  8: truth=A, pred=['A', 'C', 'D'], score=1.000
  Row  9: truth=A, pred=['A', 'B', 'C'], score=1.000
  Row 10: truth=C, pred=['C', 'A', 'D'], score=1.000
  Row 11: truth=B, pred=['B', 'C', 'A'], score=1.000
  Row 12: truth=D, pred=['D', 'A', 'B'], score=1.000
  Row 13: truth=E, pred=['E', 'D', 'B'], score=1.000
  Row 14: truth=E, pred=['E', 'A', 'D'], score=1.000
  Row 15: truth=E, pred=['E', 'B', 'C'], score=1.000
  Row 16: truth=C, pred=['C', 'B', 'E'], score=1.000
  Row 17: truth=C, pred=['C', 'B', 'E'], score=1.000
  Row 18: truth=B, pred=['B', 'D', 'A'], score=1.000
  Row 19: truth=C, pred=['C', 'D', 'A'], score=1.000

Average MAP@3 over 20 rows: 0.975

ANSWER Q8: 0.975
